# Topic 7 — Hybrid Memory: The Live Chat Meets What We Know

Combines topic 4 (Redis session) and topic 5 (Firestore profile). Requires the Redis SSH tunnel to still be open.

In [ ]:
import redis
from setup import firestore_client

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
profiles = firestore_client.collection("support_customer_profiles")

In [ ]:
def get_support_context(session_id: str, customer_id: str) -> str:
    session_turns = r.lrange(f"support_session:{session_id}:turns", 0, -1)
    profile_doc = profiles.document(customer_id).get()
    profile = profile_doc.to_dict() if profile_doc.exists else {}

    return (
        "Current chat:\n" + "\n".join(session_turns) +
        f"\n\nCustomer profile: plan={profile.get('plan_tier', 'unknown')}, "
        f"known issues={profile.get('known_issues', [])}"
    )

### One combined view SupportBot would actually use to respond

In [ ]:
r.rpush("support_session:sess_8842:turns", "customer: Still crashing, same issue as before.")
r.expire("support_session:sess_8842:turns", 1800)

print(get_support_context("sess_8842", "cust_042"))